# Ćwiczenie: Bezpieczeństwo Danych Medycznych

Miniaplikacja bezpieczeństwa danych klinicznych (Etapy 1-5):
- poufność (szyfrowanie),
- integralność (hashe i wykrywanie modyfikacji),
- pseudonimizacja/anonimizacja,
- kontrola dostępu (RBAC) i audyt,
- bezpieczeństwo modelu SI.

Cel praktyczny: zobaczyć **co, jak i po co** zabezpieczamy w danych medycznych.

## Etap 1: Przygotowanie i analiza danych

**Jak dla dziecka:**
Wyobraź sobie, że masz pudełko z kartami pacjentów. Niektóre pola to „sekrety” (np. PESEL), a inne to „parametry zdrowia” (np. ciśnienie). Najpierw musimy te pola rozdzielić, żeby wiedzieć, co chronić najmocniej.

In [1]:
import os
import uuid
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from cryptography.fernet import Fernet
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

DATA_PATH = Path("dane_pacjentow_demo.csv")
RNG = np.random.default_rng(42)

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print("Wczytano istniejący plik:", DATA_PATH)
else:
    n = 500
    age = RNG.integers(18, 90, size=n)
    sex = RNG.choice(["F", "M"], size=n)
    bmi = np.clip(RNG.normal(27, 5, size=n), 16, 50)
    sbp = np.round(95 + 0.55 * age + 1.2 * bmi + (sex == "M") * 4 + RNG.normal(0, 10, size=n)).astype(int)
    dbp = np.round(58 + 0.25 * age + 0.55 * bmi + (sex == "M") * 2 + RNG.normal(0, 6, size=n)).astype(int)
    glucose = np.clip(np.round(RNG.normal(105, 20, size=n)), 65, 280).astype(int)

    hypertension = ((sbp >= 140) | (dbp >= 90)).astype(int)
    diagnosis = np.where(hypertension == 1, "Hypertension", "Normal")

    df = pd.DataFrame(
        {
            "first_name": RNG.choice(["Anna", "Jan", "Marek", "Ewa", "Olga", "Tomasz"], size=n),
            "last_name": RNG.choice(["Nowak", "Kowalski", "Wójcik", "Mazur", "Krawczyk"], size=n),
            "pesel": [str(RNG.integers(10**10, 10**11 - 1)) for _ in range(n)],
            "phone": [f"5{RNG.integers(10000000, 99999999)}" for _ in range(n)],
            "age": age,
            "sex": sex,
            "bmi": np.round(bmi, 1),
            "systolic_bp": sbp,
            "diastolic_bp": dbp,
            "glucose": glucose,
            "diagnosis": diagnosis,
            "hypertension": hypertension,
        }
    )
    df.to_csv(DATA_PATH, index=False)
    print("Wygenerowano dane i zapisano do:", DATA_PATH)

print("\nPodgląd danych:")
print(df.head(3))

# Dane wrażliwe i kliniczne
pii_cols = ["first_name", "last_name", "pesel", "phone"]
clinical_cols = ["age", "sex", "bmi", "systolic_bp", "diastolic_bp", "glucose", "diagnosis", "hypertension"]

pii_df = df[pii_cols].copy()
clinical_df = df[clinical_cols].copy()

print("\nKolumny PII (identyfikujące):", pii_cols)
print("Kolumny kliniczne:", clinical_cols)
print("\nRozmiar zbioru:", df.shape)

Wygenerowano dane i zapisano do: dane_pacjentow_demo.csv

Podgląd danych:
  first_name last_name        pesel      phone  age sex   bmi  systolic_bp  \
0     Tomasz     Mazur  73548872414  536835612   24   F  34.2          134   
1        Ewa     Mazur  65600734274  581579110   73   M  27.5          174   
2        Jan  Kowalski  10448646876  564694385   65   M  29.9          178   

   diastolic_bp  glucose     diagnosis  hypertension  
0            86       94        Normal             0  
1            94      106  Hypertension             1  
2            95      100  Hypertension             1  

Kolumny PII (identyfikujące): ['first_name', 'last_name', 'pesel', 'phone']
Kolumny kliniczne: ['age', 'sex', 'bmi', 'systolic_bp', 'diastolic_bp', 'glucose', 'diagnosis', 'hypertension']

Rozmiar zbioru: (500, 12)


## Etap 2: Szyfrowanie i integralność danych

**Jak dla dziecka:**
- Szyfrowanie to zamknięcie informacji w sejfie.
- Klucz to jedyny pasujący kluczyk do sejfu.
- Hash to „odcisk palca” pliku: jeśli plik się zmieni, odcisk też się zmieni.

In [ ]:
KEY_PATH = Path("fernet.key")
ENCRYPTED_FILE_PATH = Path("dane_pacjentow_demo.csv.enc")
DECRYPTED_FILE_PATH = Path("dane_pacjentow_demo.dec.csv")


def save_key(path: Path) -> bytes:
    key = Fernet.generate_key()
    path.write_bytes(key)
    return key


def load_key(path: Path) -> bytes:
    return path.read_bytes()


if KEY_PATH.exists():
    key = load_key(KEY_PATH)
else:
    key = save_key(KEY_PATH)

cipher = Fernet(key)

# 2.1 Szyfrowanie wybranych kolumn
secure_df = df.copy()
for col in ["diagnosis", "pesel"]:
    secure_df[f"{col}_enc"] = secure_df[col].astype(str).apply(
        lambda x: cipher.encrypt(x.encode("utf-8")).decode("utf-8")
    )

print("Przykład szyfrowania kolumn:")
print(secure_df[["diagnosis", "diagnosis_enc", "pesel", "pesel_enc"]].head(2))

# 2.2 Hashowanie pliku i porównanie algorytmów
content = DATA_PATH.read_bytes()
hashes = {
    "sha256": hashlib.sha256(content).hexdigest(),
    "sha3_256": hashlib.sha3_256(content).hexdigest(),
    "blake2b": hashlib.blake2b(content).hexdigest(),
}

print("\nSkróty pliku (różne algorytmy):")
for name, digest in hashes.items():
    print(f"{name:10}: {digest[:24]}...{digest[-12:]}")

# 2.3 Efekt zmiany jednego bitu
mutated = bytearray(content)
mutated[0] ^= 0b00000001

orig_sha256 = hashlib.sha256(content).hexdigest()
mut_sha256 = hashlib.sha256(mutated).hexdigest()

diff_chars = sum(1 for a, b in zip(orig_sha256, mut_sha256) if a != b)
print("\nTest integralności (zmiana 1 bitu):")
print("SHA-256 oryginału:", orig_sha256)
print("SHA-256 po zmianie 1 bitu:", mut_sha256)
print("Liczba różnych znaków hex:", diff_chars, "na", len(orig_sha256))

# 2.4 Szyfrowanie całego pliku + odszyfrowanie kontrolne
encrypted_content = cipher.encrypt(content)
ENCRYPTED_FILE_PATH.write_bytes(encrypted_content)

decrypted_content = cipher.decrypt(encrypted_content)
DECRYPTED_FILE_PATH.write_bytes(decrypted_content)

print("\nSzyfrowanie całego pliku:")
print("Plik zaszyfrowany:", ENCRYPTED_FILE_PATH)
print("Plik odszyfrowany:", DECRYPTED_FILE_PATH)
print("Czy odszyfrowany = oryginalny?", decrypted_content == content)

## Etap 3: Pseudonimizacja i anonimizacja

**Jak dla dziecka:**
- Pseudonimizacja: zamieniamy imię i nazwisko na losowy numer (nadal da się odtworzyć mapę, jeśli ją trzymasz osobno).
- Anonimizacja: usuwamy lub uogólniamy dane tak, żeby trudniej było wskazać konkretną osobę.

In [ ]:
pseudo_df = df.copy()

# 3.1 Pseudonimizacja
pseudo_df["patient_id"] = [str(uuid.uuid4()) for _ in range(len(pseudo_df))]

id_map_df = pseudo_df[["patient_id", "first_name", "last_name", "pesel", "phone"]].copy()
clinical_with_pid = pseudo_df.drop(columns=["first_name", "last_name", "pesel", "phone"]).copy()

print("Przykład pseudonimizacji (patient_id):")
print(clinical_with_pid[["patient_id", "age", "bmi", "diagnosis"]].head(3))

# 3.2 Anonimizacja (generalizacja + maskowanie + usuwanie identyfikatorów)
def age_group(age: int) -> str:
    if age < 30:
        return "<30"
    if age < 45:
        return "30-44"
    if age < 60:
        return "45-59"
    if age < 75:
        return "60-74"
    return "75+"

anon_df = pseudo_df.copy()
anon_df["age_group"] = anon_df["age"].apply(age_group)
anon_df["bmi_round"] = anon_df["bmi"].round(0)

# Maskowanie telefonu (zostawiamy tylko prefiks)
anon_df["phone_masked"] = anon_df["phone"].astype(str).str[:3] + "******"

# Usuwamy kolumny identyfikujące i zbyt szczegółowe
anon_df = anon_df.drop(columns=["first_name", "last_name", "pesel", "phone", "age", "bmi"])

print("\nPrzykład anonimizacji:")
print(anon_df.head(3))

# 3.3 Wpływ anonimizacji na model predykcyjny
def evaluate_model(input_df: pd.DataFrame, target_col: str = "hypertension"):
    model_df = input_df.copy()
    y = model_df[target_col].astype(int)
    X = model_df.drop(columns=[target_col, "diagnosis", "patient_id", "diagnosis_enc", "pesel_enc"], errors="ignore")

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler()),
                ]),
                num_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore")),
                ]),
                cat_cols,
            ),
        ]
    )

    clf = Pipeline([
        ("pre", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    return {
        "acc": accuracy_score(y_test, pred),
        "auc": roc_auc_score(y_test, proba),
        "n_features": X.shape[1],
    }

baseline_metrics = evaluate_model(df)
anon_metrics = evaluate_model(anon_df)

print("\nPorównanie jakości modelu:")
print(f"Przed anonimizacją: ACC={baseline_metrics['acc']:.4f}, AUC={baseline_metrics['auc']:.4f}, cechy={baseline_metrics['n_features']}")
print(f"Po anonimizacji:    ACC={anon_metrics['acc']:.4f}, AUC={anon_metrics['auc']:.4f}, cechy={anon_metrics['n_features']}")

print("\nInterpretacja:")
print("- Jeśli metryki spadły mało: anonimizacja dobrze chroni dane i prawie nie psuje modelu.")
print("- Jeśli spadły mocno: trzeba poprawić technikę anonimizacji (mniejsza utrata informacji).")

## Etap 4: Kontrola dostępu (RBAC) i audyt

**Jak dla dziecka:**
Każdy dostaje inny klucz do drzwi:
- administrator widzi wszystko,
- lekarz widzi dane do leczenia,
- analityk widzi tylko zanonimizowane dane.
A do zeszytu zapisujemy, kto i kiedy otwierał drzwi (audyt).

In [ ]:
from datetime import datetime

audit_log = []

ROLES = {
    "administrator": {
        "columns": clinical_with_pid.columns.tolist(),
    },
    "lekarz": {
        "columns": ["patient_id", "age", "sex", "bmi", "systolic_bp", "diastolic_bp", "glucose", "diagnosis", "hypertension"],
    },
    "analityk": {
        "columns": anon_df.columns.tolist(),
        "source": "anon",
    },
}


def add_audit(user: str, role: str, action: str, status: str):
    audit_log.append(
        {
            "time": datetime.now().isoformat(timespec="seconds"),
            "user": user,
            "role": role,
            "action": action,
            "status": status,
        }
    )


def get_role_view(user: str, role: str) -> pd.DataFrame:
    if role not in ROLES:
        add_audit(user, role, "ACCESS", "DENIED_UNKNOWN_ROLE")
        raise ValueError(f"Nieznana rola: {role}")

    cfg = ROLES[role]
    source_df = anon_df if cfg.get("source") == "anon" else clinical_with_pid
    allowed = [c for c in cfg["columns"] if c in source_df.columns]

    add_audit(user, role, f"READ_COLUMNS:{','.join(allowed)}", "GRANTED")
    return source_df[allowed].copy()


admin_view = get_role_view("alice", "administrator")
doctor_view = get_role_view("bob", "lekarz")
analyst_view = get_role_view("carol", "analityk")

print("Widok administratora (2 wiersze):")
print(admin_view.head(2))
print("\nWidok lekarza (2 wiersze):")
print(doctor_view.head(2))
print("\nWidok analityka (2 wiersze):")
print(analyst_view.head(2))

print("\nLog audytu (ostatnie wpisy):")
print(pd.DataFrame(audit_log).tail(10))

## Etap 5: Bezpieczeństwo modelu SI

**Jak dla dziecka:**
Zanim model coś policzy, sprawdzamy trzy rzeczy:
1. czy plik jest ten sam co wcześniej (integralność),
2. czy dane wejściowe mają sens (walidacja),
3. czy ktoś nie próbował oszukać systemu (manipulacja / poisoning / adversarial).

In [ ]:
# 5.1 Hash referencyjny i detekcja naruszeń integralności
TRAIN_HASH = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()


def verify_file_integrity(path: Path, expected_hash: str) -> bool:
    return hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash


print("Integralność danych przed inferencją:", verify_file_integrity(DATA_PATH, TRAIN_HASH))

# 5.2 Walidacja wejścia
ALLOWED_SEX = {"F", "M"}


def validate_row(row: pd.Series) -> tuple[bool, str]:
    checks = [
        (18 <= row.get("age", -1) <= 100, "wiek poza zakresem"),
        (row.get("sex") in ALLOWED_SEX, "niepoprawna płeć"),
        (12 <= row.get("bmi", -1) <= 65, "BMI poza zakresem"),
        (70 <= row.get("systolic_bp", -1) <= 260, "SBP poza zakresem"),
        (40 <= row.get("diastolic_bp", -1) <= 160, "DBP poza zakresem"),
        (50 <= row.get("glucose", -1) <= 450, "glukoza poza zakresem"),
    ]
    for ok, msg in checks:
        if not ok:
            return False, msg
    return True, "OK"


# Trening prostego modelu referencyjnego
model_features = ["age", "sex", "bmi", "systolic_bp", "diastolic_bp", "glucose"]
X = df[model_features].copy()
y = df["hypertension"].astype(int)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

pre = ColumnTransformer(
    [
        (
            "num",
            Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]),
            num_cols,
        ),
        (
            "cat",
            Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]),
            cat_cols,
        ),
    ]
)

secure_model = Pipeline([
    ("pre", pre),
    ("model", LogisticRegression(max_iter=1000)),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
secure_model.fit(X_train, y_train)


def secure_predict(input_row: pd.Series) -> tuple[int, float]:
    if not verify_file_integrity(DATA_PATH, TRAIN_HASH):
        raise RuntimeError("Naruszenie integralności: plik danych został zmieniony.")

    ok, msg = validate_row(input_row)
    if not ok:
        raise ValueError(f"Walidacja wejścia nieudana: {msg}")

    row_df = input_row[model_features].to_frame().T
    proba = float(secure_model.predict_proba(row_df)[0, 1])
    pred = int(proba >= 0.5)
    return pred, proba


# 5.3 Test bezpiecznej predykcji
sample = df.iloc[0]
pred, proba = secure_predict(sample)
print("\nBezpieczna predykcja dla pacjenta 0:", pred, "| P(hypertension)=", round(proba, 4))

# 5.4 Symulacja manipulacji danymi (data tampering) i detekcja
original_bytes = DATA_PATH.read_bytes()
tampered_bytes = original_bytes + b"\n#TAMPERED"
DATA_PATH.write_bytes(tampered_bytes)

print("Integralność po manipulacji:", verify_file_integrity(DATA_PATH, TRAIN_HASH))

# Przywrócenie oryginalnego pliku
DATA_PATH.write_bytes(original_bytes)
print("Integralność po przywróceniu:", verify_file_integrity(DATA_PATH, TRAIN_HASH))

# 5.5 Prosty przykład ryzyk AI
risks = {
    "poisoning": "Atakujący dodaje fałszywe dane do treningu, aby wypaczyć model.",
    "data_tampering": "Modyfikacja danych wejściowych lub plików między treningiem a inferencją.",
    "adversarial_examples": "Minimalna zmiana danych wejściowych powoduje błędną predykcję.",
}

print("\nGłówne ryzyka bezpieczeństwa AI:")
for k, v in risks.items():
    print(f"- {k}: {v}")

## Etap 6: Audyt logów i ulepszenia polityki bezpieczeństwa

**Jak dla dziecka:**
To jak dziennik w szkole: kto wszedł do klasy, kiedy, i czy miał pozwolenie. Dzięki temu łatwo zauważyć podejrzane zachowanie.

In [ ]:
audit_df = pd.DataFrame(audit_log)
print("Liczba wpisów audytowych:", len(audit_df))

if not audit_df.empty:
    print("\nStatusy dostępu:")
    print(audit_df["status"].value_counts())

    print("\nAktywność użytkowników:")
    print(audit_df.groupby(["user", "role"]).size().sort_values(ascending=False))

    denied = audit_df[audit_df["status"].str.contains("DENIED", na=False)]
    print("\nPodejrzane zdarzenia (DENIED):", len(denied))

print("\nProponowane ulepszenia polityki bezpieczeństwa:")
recommendations = [
    "Wymusić MFA dla ról uprzywilejowanych (administrator, lekarz).",
    "Dodać automatyczne alerty po >=3 nieudanych próbach dostępu.",
    "Rotować klucz szyfrujący cyklicznie (np. co 30 dni).",
    "Przechowywać klucze poza repozytorium (np. vault, HSM).",
    "Podpisać model i dane treningowe podpisem cyfrowym.",
    "Ustawić retencję logów i regularny przegląd audytowy.",
]
for idx, rec in enumerate(recommendations, start=1):
    print(f"{idx}. {rec}")

## Krótkie wnioski końcowe

1. Szyfrowanie (kolumn i całego pliku) skutecznie chroni poufność danych medycznych.
2. Hashowanie (SHA-256/SHA-3/BLAKE2) pozwala szybko wykryć każdą manipulację danymi.
3. Pseudonimizacja i anonimizacja zwiększają prywatność, zwykle przy niewielkim koszcie jakości modelu.
4. RBAC + audyt dają kontrolę nad tym, kto widzi jakie dane i kiedy.
5. Bezpieczna inferencja (integralność + walidacja wejścia) ogranicza ryzyko ataków typu data tampering.

To ćwiczenie pokazuje, że bezpieczeństwo w AI medycznym to nie jeden „trik”, tylko zestaw warstw ochrony działających razem.